# Medical Data With Built-In ifcfill And SeqTrees

This notebook demonstrates a fuller workflow:

1. Load public Synthea healthcare sample data from a URL.
2. Keep a patient-level table with dates, categorical values, numeric values, and missing values.
3. Fit SeqTrees directly on the mixed pandas DataFrame. SeqTrees uses `ifcfill` internally with label encoding to prepare the modelling table.
4. Sample the same number of rows. SeqTrees restores generated rows back to patient-like DataFrame values through the fitted internal transformer.

The Synthea sample data is generated healthcare data, not real patient PHI. That makes it suitable for a public tutorial while still looking like a medical table.

In [ ]:
# Run the following in a fresh notebook environment.
# %pip install -q "seqtrees[examples]"


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

import pandas as pd
from IPython.display import display


# When running this notebook from the repository checkout, prefer the local SeqTrees source tree.
repo_root = Path.cwd()
if (repo_root / "src" / "seqtrees").exists():
    sys.path.insert(0, str(repo_root / "src"))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)

from seqtrees import SequentialTreeSynthesizer


In [ ]:
DATA_URL = "https://github.com/synthetichealth/synthea-sample-data/raw/main/downloads/synthea_sample_data_csv_apr2020.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "synthea_sample_data_csv_apr2020.zip"

DATA_DIR.mkdir(exist_ok=True)
if not ZIP_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with ZipFile(ZIP_PATH) as zf:
    csv_names = zf.namelist()
    patients_name = next(name for name in csv_names if name.endswith("patients.csv"))
    with zf.open(patients_name) as f:
        raw_patients = pd.read_csv(f)

print(f"Raw patients: {raw_patients.shape[0]:,} rows x {raw_patients.shape[1]:,} columns")
display(raw_patients.head())

In [ ]:
# A compact but mixed-type patient table: dates, strings, floats, and missing DEATHDATE values.
columns = [
    "BIRTHDATE",
    "DEATHDATE",
    "MARITAL",
    "RACE",
    "ETHNICITY",
    "GENDER",
    "BIRTHPLACE",
    "CITY",
    "STATE",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
]

patients = raw_patients[columns].copy()
patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"], errors="coerce")
patients["DEATHDATE"] = pd.to_datetime(patients["DEATHDATE"], errors="coerce")

# Keep the example quick for notebooks while preserving enough rows for synthesis.
patients = patients.sample(min(len(patients), 1000), random_state=42).reset_index(drop=True)

display(patients.head())

missing_counts = patients.isna().sum().sort_values(ascending=False).head(8)
display(missing_counts.rename("missing_count").to_frame())

SeqTrees can now fit the mixed `patients` DataFrame directly. During `fit`, it creates an internal `ifcfill.IFCTransformer` with label encoding, imputes missing values, converts datetimes into numeric offsets for modelling, and records the source column types. The exact model-ready table is available with `model.get_preprocessed_data()`.

In [ ]:
model = SequentialTreeSynthesizer(
    optimize_order=True,
    tree_backend="auto",
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

model.fit(patients)

print("Variable order:", model.get_variable_order())

display(model.ifc_transformer_.missing_report_.sort_values("missing_fraction", ascending=False).head(10))
display(pd.Series(model.ifc_column_types_, name="ifc_column_type").to_frame())


In [ ]:
preprocessed_patients = model.get_preprocessed_data()
display(preprocessed_patients.head())
preprocessed_patients.info()


SeqTrees fitted on `patients` directly, but the internal modelling table is numeric. Categorical columns are label-encoded and sampled only from observed labels, while source integer columns are treated as numeric integer variables. The final `sample` call restores the generated data through the fitted internal transformer.

In [ ]:
synthetic_patients = model.sample(len(patients), random_state=101, as_dataframe=True, n_jobs=-1)

print(f"Synthetic patients: {synthetic_patients.shape[0]:,} rows x {synthetic_patients.shape[1]:,} columns")
display(synthetic_patients.head())


In [ ]:
column_summary = pd.DataFrame(
    {
        "ifc_column_type": pd.Series(model.ifc_column_types_),
        "original_dtype": patients.dtypes.astype(str),
        "synthetic_dtype": synthetic_patients.dtypes.astype(str),
    }
)

display(column_summary)


In [ ]:
summary = pd.DataFrame(
    {
        "original_missing": patients.isna().mean(),
        "synthetic_missing": synthetic_patients.isna().mean(),
    }
)

display(summary.sort_values("original_missing", ascending=False))

This example keeps the same row count as the input table. For real projects, inspect utility and privacy risk before publishing synthetic data, especially when the source resembles patient-level records.